In [3]:
!pip install -q langchain-core langchain-text-splitters langchain-huggingface langchain-chroma chromadb "langchain-docling[local]" langsmith langchain-groq langchain-community pypdf pandas

In [5]:
import os
import pandas as pd
from dotenv import load_dotenv
from langchain_core.documents import Document
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma
from langchain_groq import ChatGroq
from langchain_core.tools import tool
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.messages import HumanMessage, ToolMessage

In [9]:
%pip install sentence-transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 780.3/780.3 kB 1.3 MB/s  0:00:00 eta 0:00:01
Note: you may need to restart the kernel to use updated packages.


# Q1 - Load the budget PDF and split the text

In [6]:
budget_pages = PyPDFLoader("budget_speech.pdf").load()

for page_item in budget_pages:
    page_item.metadata["page"] = page_item.metadata.get("page", 0) + 1
    page_item.page_content = " ".join(
        page_item.page_content.replace("-\n", "").split()
    )

text_segments = RecursiveCharacterTextSplitter(
    chunk_size=800,
    chunk_overlap=150
).split_documents(budget_pages)


def sanitize_document_metadata(document_item: Document) -> Document:
    safe_metadata = {
        k: v for k, v in document_item.metadata.items()
        if isinstance(v, (str, int, float, bool))
    }
    return Document(
        page_content=document_item.page_content,
        metadata=safe_metadata
    )


prepared_segments = [
    sanitize_document_metadata(page_item)
    for page_item in text_segments
]

print(f"Q1 Chunks: {len(prepared_segments)}")
print(f"Metadata: {prepared_segments[0].metadata}")

Q1 Chunks: 172
Metadata: {'producer': 'doPDF Ver 8.4 Build 935', 'creator': 'Adobe Acrobat (64-bit) 25.1.20997', 'creationdate': '2026-02-01T05:37:46+05:30', 'moddate': '2026-02-01T05:39:37+05:30', 'title': '', 'source': 'budget_speech.pdf', 'total_pages': 65, 'page': 1, 'page_label': '1'}


# Q2 - Create the vector database


In [10]:
embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-mpnet-base-v2"
)

budget_vector_db = Chroma.from_documents(
    documents=prepared_segments,
    embedding=embedding_model,
    persist_directory="./budget_vector_db"
)

print("\nQ2 Top 3 Results:")

results = budget_vector_db.similarity_search_with_score(
    "Biopharma SHAKTI outlay", k=3
)

for document_item, distance_score in results:
    print(
        f"Score: {distance_score:.4f} | "
        f"Page {document_item.metadata['page']}: "
        f"{document_item.page_content[:120]}..."
    )

/home/luna_esther/Documents/SCFP125036_3rd_Sem/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 2695.67it/s]



Q2 Top 3 Results:
Score: 0.5969 | Page 8: 4 Scaling up manufacturing in 7 strategic and frontier sectors: 16. Biopharma SHAKTI (Strategy for Healthcare Advancemen...
Score: 0.8913 | Page 8: network with 3 new National Institutes of Pharmaceutical Education and Research (NIPER) and upgrading 7 existing ones. I...
Score: 1.0237 | Page 59: Ministry of Defence As applicable Nil VII. Drugs/ Medicines 10. 17 new drugs/medicines to be added in List 3 appended to...


# Q3 and Q4 - Create tools and connect the agent

In [11]:

@tool
def retrieve_budget_context(question_text: str) -> str:
    """Search the Budget 2026-27 speech for schemes, outlays, tax rates, and fiscal numbers."""

    results = budget_vector_db.similarity_search(question_text, k=4)

    return "\n".join(
        f"[Page {page_item.metadata['page']}] {page_item.page_content}"
        for page_item in results
    )


@tool
def calculate_expression(math_expression: str) -> str:
    """Calculate simple math expressions."""

    return str(round(eval(math_expression, {"__builtins__": {}}), 2))


available_tools = {
    "retrieve_budget_context": retrieve_budget_context,
    "calculate_expression": calculate_expression
}

load_dotenv()

GROQ_API_KEY = os.getenv("GROQ_API_KEY")

if not GROQ_API_KEY:
    raise ValueError("GROQ_API_KEY not found. Check your .env file.")

chat_model = ChatGroq(
    model="openai/gpt-oss-120b",
    api_key=GROQ_API_KEY,
    temperature=0
)

agent_prompt = ChatPromptTemplate.from_messages([
    ("system", 
     "Answer greetings/general concepts directly without tools. "
     "Use retrieve_budget_context for budget facts and cite [Page X]. "
     "If not in the document, say so. "
     "Use calculate_expression for math."),
    MessagesPlaceholder("conversation_history")
])

agent_pipeline = agent_prompt | chat_model.bind_tools(
    list(available_tools.values())
)

print("Q3 & Q4: Tools and agent are ready!")

Q3 & Q4: Tools and agent are ready!


# Q4 and Q5 - Run the test questions


In [ ]:


test_prompts = [
    "What is the outlay of Biopharma SHAKTI?",
    "How does the 2026-27 fiscal deficit compare with 2025-26?",
    "What changes for NCCD on chewing tobacco, and what is the effective duty?",
    "Hi, how are you?",
    "Explain fiscal deficit in simple words.",
    "What are the new income tax slab rates in this budget?",
    "By what percentage does public capex rise from BE 2025-26 to FY2026-27?",
    "What is the combined outlay of Biopharma SHAKTI, the Container Manufacturing scheme and CCUS?",
    "exit"
]

prompt_iterator = iter(test_prompts)
evaluation_records = []

while True:
    current_prompt = next(prompt_iterator)

    if current_prompt.lower() == "exit":
        break

    conversation_history = [HumanMessage(content=current_prompt)]
    invoked_tools = []

    for _ in range(5):
        assistant_reply = agent_pipeline.invoke({
            "conversation_history": conversation_history
        })

        conversation_history.append(assistant_reply)

        if not assistant_reply.tool_calls:
            break

        for tool_request in assistant_reply.tool_calls:
            invoked_tools.append(tool_request["name"])

            tool_result = available_tools[tool_request["name"]].invoke(
                tool_request["args"]
            )

            conversation_history.append(
                ToolMessage(
                    content=tool_result,
                    tool_call_id=tool_request["id"]
                )
            )

    final_response = conversation_history[-1].content

    print(f"\nQ: {current_prompt}")
    print(f"Tools: {invoked_tools}")
    print(f"A: {final_response}")

    evaluation_records.append({
        "Query": current_prompt,
        "Searched (Y/N)": "Y" if "retrieve_budget_context" in invoked_tools else "N",
        "Tools Used": ", ".join(invoked_tools) or "None",
        "Final Answer": final_response
    })


evaluation_df = pd.DataFrame(evaluation_records)

display(evaluation_df)

# Check the expected tool selection
expected_search = [True, True, True, False, False, True, True, True]

correct_routes = sum(
    (record["Searched (Y/N)"] == "Y") == expected
    for record, expected in zip(evaluation_records, expected_search)
)

print(
    f"\nTool Selection Check: {correct_routes}/"
    f"{len(expected_search)}"
)


Q: What is the outlay of Biopharma SHAKTI?
Tools: ['retrieve_budget_context']
A: The Biopharma SHAKTI programme is proposed with an outlay of **₹ 10,000 crores** over the next five years【Page 8】.

Q: How does the 2026-27 fiscal deficit compare with 2025-26?
Tools: ['retrieve_budget_context']
A: The Budget speech states that the fiscal deficit for **FY 2026‑27 is projected at 4.3 % of GDP**【Page 23】.  

For **FY 2025‑26 the budget estimate was 4.4 % of GDP**【Page 23】.  

**Comparison:** The 2026‑27 deficit is therefore **0.1 percentage point lower** than the 2025‑26 deficit, reflecting a modest improvement in fiscal balance.

Q: What changes for NCCD on chewing tobacco, and what is the effective duty?
Tools: ['retrieve_budget_context']
A: The budget proposes to raise the **National Calamity Contingent Duty (NCCD) rate for chewing tobacco from 25 % to 60 %**. However, a separate notification will keep the **effective duty rate at 25 %** despite the higher NCCD rate — the higher rate is 

,Query,Searched (Y/N),Tools Used,Final Answer
0,What is the outlay of Biopharma SHAKTI?,Y,retrieve_budget_context,The Biopharma SHAKTI programme is proposed wit...
1,How does the 2026-27 fiscal deficit compare wi...,Y,retrieve_budget_context,The Budget speech states that the fiscal defic...
2,"What changes for NCCD on chewing tobacco, and ...",Y,retrieve_budget_context,The budget proposes to raise the **National Ca...
3,"Hi, how are you?",N,None,"Hello! I'm doing well, thank you. How can I as..."
4,Explain fiscal deficit in simple words.,N,None,A fiscal deficit happens when a government spe...
5,What are the new income tax slab rates in this...,Y,"retrieve_budget_context, retrieve_budget_conte...",[Page 29] 0.02 percent. STT on options premium...
6,By what percentage does public capex rise from...,Y,"retrieve_budget_context, calculate_expression",Public capital expenditure rises from **₹11.2 ...
7,What is the combined outlay of Biopharma SHAKT...,Y,"retrieve_budget_context, retrieve_budget_conte...",The three programmes have the following budget...



Tool Selection Check: 8/8
